# Umubyeyi
This is the single reproducible notebook for the capstone. It covers data provenance,
missing-value handling, leakage prevention, seven-model screening experiments, model
selection, confusion matrices, explainability, bilingual retrieval, LoRA fine-tuning,
loss curves, generator comparison, language-specific quality gates, and artifact export.

**Runtime:** In Google Colab choose **Runtime → Change runtime type → T4 GPU**, then use
**Runtime → Run all**. Classical ML runs on CPU; the generator section requires CUDA.

The notebook clones the public `dev` branch when project data is not already present.
No API key or database credential is required, and no patient conversation is uploaded.


In [ ]:
# Colab/runtime dependencies. Re-running this cell is safe.
%pip install -q 'scikit-learn==1.9.0' 'transformers==4.46.3'         'datasets==3.1.0' 'peft==0.13.2' 'accelerate>=1.1'         'sentencepiece>=0.2' 'rouge-score>=0.1.2' joblib pandas matplotlib


In [ ]:
from pathlib import Path
import json, os, random, re, shutil, subprocess, sys, time, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore', category=UserWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path('/content/UMUBYEYI')]
    for candidate in candidates:
        if (candidate / 'data/postpartum_depression/PPD_dataset_v3.csv').exists():
            return candidate
    destination = Path('/content/UMUBYEYI')
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'dev',
        'https://github.com/IrutingaboRaissa/UMUBYEYI.git', str(destination)
    ], check=True)
    return destination

ROOT = find_project_root()
DATA = ROOT / 'data/postpartum_depression/PPD_dataset_v3.csv'
DICTIONARY = ROOT / 'data/postpartum_depression/PPD_Data_Dictionary_v3.csv'
KNOWLEDGE = ROOT / 'data/knowledge/postpartum_wellbeing.json'
OUTPUT_ROOT = Path('/content/umubyeyi_outputs') if Path('/content').exists() else ROOT / 'notebook_outputs'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
MODEL_DIR = OUTPUT_ROOT / 'models'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT)
print('Outputs:', OUTPUT_ROOT)


## 1. Data Audit

The dataset is *Data for Postpartum Depression Prediction in Bangladesh*, Mendeley Data version 3, DOI `10.17632/4nznnrk8cg.3`, licensed CC BY 4.0. It covers Bangladesh and up to 24 months postpartum; therefore it is not clinical validation for Rwanda or specifically first-time mothers.

In [ ]:
df = pd.read_csv(DATA)
dictionary = pd.read_csv(DICTIONARY)
audit = pd.Series({
    'rows': len(df),
    'columns': len(df.columns),
    'duplicate rows': int(df.duplicated().sum()),
    'missing cells': int(df.isna().sum().sum()),
    'EPDS High': int((df['EPDS Result'] == 'High').sum()),
    'EPDS Medium': int((df['EPDS Result'] == 'Medium').sum()),
    'EPDS Low': int((df['EPDS Result'] == 'Low').sum()),
})
display(audit.to_frame('value'))
display(dictionary.head(10))

## 2. Exploratory Analysis

These plots describe the dataset before modelling. Missing predictor values are handled inside each training pipeline, preventing information from the validation or test sets leaking into preprocessing.

In [ ]:
target_binary = df['EPDS Result'].eq('High').map({True: 'elevated', False: 'not_elevated'})
COLORS = {'elevated': '#704f6f', 'not_elevated': '#d8a48f'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epds_counts = df['EPDS Result'].value_counts().reindex(['High', 'Medium', 'Low'])
axes[0, 0].bar(epds_counts.index, epds_counts.values, color=['#704f6f', '#a77887', '#d8a48f'])
axes[0, 0].set(title='Original three EPDS result categories', ylabel='Participants')
for container in axes[0, 0].containers: axes[0, 0].bar_label(container)

binary_counts = target_binary.value_counts().reindex(['elevated', 'not_elevated'])
axes[0, 1].bar(['Elevated', 'Not elevated'], binary_counts, color=[COLORS[x] for x in binary_counts.index])
axes[0, 1].set(title='Binary modelling target', ylabel='Participants')
for container in axes[0, 1].containers: axes[0, 1].bar_label(container)

missing = df.isna().sum().sort_values(ascending=False).head(15).sort_values()
missing.plot.barh(ax=axes[1, 0], color='#a77887')
axes[1, 0].set(title='Top 15 columns by missing values', xlabel='Missing cells')

age = pd.to_numeric(df['Age'], errors='coerce')
for label in ['elevated', 'not_elevated']:
    axes[1, 1].hist(age[target_binary == label].dropna(), bins=12, alpha=.65,
                    label=label.replace('_', ' '), color=COLORS[label])
axes[1, 1].set(title='Age distribution by target', xlabel='Age', ylabel='Participants')
axes[1, 1].legend()
plt.tight_layout(); plt.show()

**Interpretation:** The original target has 350 High, 190 Medium and 260 Low records. Combining Low and Medium creates 350 elevated versus 450 not-elevated cases, so the classes are reasonably balanced. Missingness is substantial in some predictors and is therefore handled inside the pipelines. The overlapping age distributions show that age alone cannot separate the two classes.

## 3. Feature Selection

The positive class is `EPDS Result == High`. Low and Medium become `not_elevated`. All concurrent EPDS/PHQ questionnaire items, their totals and derived results are excluded. Otherwise, a model could reconstruct the questionnaire score rather than learn from prior/contextual risk factors.

In [ ]:
TARGET = 'EPDS Result'
FIRST_SCALE_ITEM = 'Little interest or pleasure in doing things'
scale_start = df.columns.get_loc(FIRST_SCALE_ITEM)
excluded_columns = set(df.columns[scale_start:]) | {'sr', TARGET}
FEATURES = [column for column in df.columns if column not in excluded_columns]
X = df[FEATURES].copy()
y = target_binary.copy()

print('Predictor columns retained:', len(FEATURES))
print('Concurrent questionnaire/derived columns excluded:', len(excluded_columns))
display(pd.DataFrame({'retained predictor': FEATURES}).head(46))

## 4. Data Split

- Training: 70% (560 rows), used to fit candidate models.
- Validation: 15% (120 rows), used to compare algorithms and select the winner.
- Test: 15% (120 rows), untouched until after model selection.
- Seed: 42, making the split reproducible.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=.50, stratify=y_temp, random_state=SEED)

split_table = pd.DataFrame({
    'rows': [len(X_train), len(X_val), len(X_test)],
    'elevated': [int((y_train == 'elevated').sum()), int((y_val == 'elevated').sum()), int((y_test == 'elevated').sum())],
    'not elevated': [int((y_train == 'not_elevated').sum()), int((y_val == 'not_elevated').sum()), int((y_test == 'not_elevated').sum())],
}, index=['Train', 'Validation', 'Test'])
display(split_table)
ax = split_table[['elevated', 'not elevated']].plot.bar(stacked=True, figsize=(8, 5),
    color=['#704f6f', '#d8a48f'], title='Stratified 70/15/15 split')
ax.set(ylabel='Rows', xlabel='Split'); ax.tick_params(axis='x', rotation=0)
plt.show()

**Interpretation:** Stratification preserves nearly the same elevated/not-elevated proportion in training, validation and test data. The independent 120-row test set is not used to choose a model.

## 5. Preprocessing and Metrics

Numerical columns use median imputation and standardisation. Categorical columns use most-frequent imputation and one-hot encoding. Each pipeline learns preprocessing only from its training input.

For the positive `elevated` class:

- Accuracy = `(TP + TN) / all cases`
- Precision = `TP / (TP + FP)`
- Recall = `TP / (TP + FN)`
- F1 = harmonic balance of precision and recall

RMSE is not appropriate because this is classification, not regression.

In [ ]:
def make_pipeline(model, feature_columns):
    numeric = [c for c in ['Age', 'Number of the latest pregnancy'] if c in feature_columns]
    categorical = [c for c in feature_columns if c not in numeric]
    preprocess = ColumnTransformer([
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), numeric),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=2)),
        ]), categorical),
    ])
    return Pipeline([('preprocess', preprocess), ('model', model)])

def calculate_metrics(model, features, labels):
    predictions = model.predict(features)
    order = ['elevated', 'not_elevated']
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, pos_label='elevated', zero_division=0),
        'recall': recall_score(labels, predictions, pos_label='elevated', zero_division=0),
        'f1_score': f1_score(labels, predictions, pos_label='elevated', zero_division=0),
        'confusion_matrix': confusion_matrix(labels, predictions, labels=order),
    }

def model_definitions():
    return {
        'Dummy baseline': DummyClassifier(strategy='most_frequent'),
        'Logistic Regression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=SEED),
        'Random Forest': RandomForestClassifier(n_estimators=500, min_samples_leaf=3, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
        'Extra Trees': ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3, class_weight='balanced', n_jobs=-1, random_state=SEED),
        'Support Vector Machine': SVC(C=1.0, kernel='rbf', class_weight='balanced', random_state=SEED),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=11, weights='distance'),
    }

## 6. Full Model Comparison

Training time is recorded to demonstrate why classical models finish quickly on 560 rows. Model selection uses validation F1 for the elevated-risk class.

In [ ]:
full_models, full_results = {}, {}
for name, estimator in model_definitions().items():
    pipeline = make_pipeline(estimator, FEATURES)
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    result = calculate_metrics(pipeline, X_val, y_val)
    result['fit_seconds'] = elapsed
    full_models[name] = pipeline
    full_results[name] = result

full_comparison = pd.DataFrame({name: {
    'accuracy': value['accuracy'], 'precision': value['precision'],
    'recall': value['recall'], 'f1_score': value['f1_score'],
    'fit_seconds': value['fit_seconds']
} for name, value in full_results.items()}).T
display(full_comparison.sort_values('f1_score', ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
full_comparison[['accuracy', 'precision', 'recall', 'f1_score']].plot.bar(
    ax=axes[0], color=['#4f6d7a', '#d8a48f', '#704f6f', '#8fb996'])
axes[0].set(title='Full model: validation metrics', ylabel='Score', ylim=(0, 1), xlabel='Model')
axes[0].tick_params(axis='x', rotation=35)
full_comparison['fit_seconds'].sort_values().plot.barh(ax=axes[1], color='#4f6d7a')
axes[1].set(title='Training time on 560 rows', xlabel='Seconds', ylabel='Model')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(17, 9))
for ax, (name, result) in zip(axes.flat, full_results.items()):
    matrix = result['confusion_matrix']
    ax.imshow(matrix, cmap='Purples', vmin=0, vmax=max(1, matrix.max()))
    for row in range(2):
        for col in range(2): ax.text(col, row, matrix[row, col], ha='center', va='center', fontsize=12)
    ax.set_xticks([0, 1], ['Elevated', 'Not elevated'])
    ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
    ax.set(title=name, xlabel='Predicted', ylabel='Actual')
axes.flat[-1].axis('off')
fig.suptitle('Validation confusion matrix for every full-model candidate', fontsize=15)
plt.tight_layout(); plt.show()

**Interpretation:** Random Forest provides the strongest validation F1-score, while the dummy model never identifies an elevated case. The timing chart explains why training is quick: even the ensemble models take only seconds on 560 rows. The confusion matrices show the exact correct and incorrect validation predictions behind each metric.

## 7. Full Model Evaluation

Random Forest wins by validation F1. Its configuration is refitted using train + validation rows, then evaluated once on the untouched test rows.

In [ ]:
full_winner_name = full_comparison['f1_score'].idxmax()
X_fit, y_fit = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
full_winner = make_pipeline(model_definitions()[full_winner_name], FEATURES)
full_winner.fit(X_fit, y_fit)
full_test = calculate_metrics(full_winner, X_test, y_test)

display(pd.Series({k: v for k, v in full_test.items() if k != 'confusion_matrix'},
                  name='untouched test score').round(4).to_frame())
cm = full_test['confusion_matrix']
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Greens')
for row in range(2):
    for col in range(2): ax.text(col, row, cm[row, col], ha='center', va='center', fontsize=15)
ax.set_xticks([0, 1], ['Elevated', 'Not elevated']); ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
ax.set(title=f'Untouched test confusion matrix: {full_winner_name}', xlabel='Predicted', ylabel='Actual')
plt.show()

tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
manual_check = pd.Series({
    'accuracy': (tp + tn) / cm.sum(), 'precision': tp / (tp + fp),
    'recall': tp / (tp + fn), 'f1_score': 2 * tp / (2 * tp + fp + fn)
}, name='manually recomputed from confusion matrix')
display(manual_check.round(4).to_frame())

**Interpretation:** On 120 untouched cases, Random Forest correctly identifies 43 of 53 elevated cases and misses 10. It also produces 13 false elevated-risk alerts. The manually recomputed values confirm that the reported accuracy, precision, recall and F1 come directly from these counts.

In [ ]:
# Explain the selected Random Forest using its transformed feature importances.
feature_names = full_winner.named_steps['preprocess'].get_feature_names_out()
importances = full_winner.named_steps['model'].feature_importances_
importance_table = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)
ax = importance_table.sort_values().plot.barh(figsize=(10, 7), color='#704f6f',
    title='Random Forest: top 20 transformed feature importances')
ax.set(xlabel='Impurity-based importance', ylabel='Transformed predictor')
plt.tight_layout(); plt.show()

**Interpretation:** The importance chart shows which transformed predictors most influenced Random Forest decisions. Importance indicates predictive contribution within this dataset; it does not prove that a factor causes postpartum depression.

## 8. Check-in Model Comparison

The deployed check-in cannot reasonably ask 46 questions. This second experiment repeats the full procedure using 15 understandable inputs. The same original split indices and seven algorithms are used.

In [ ]:
CHECKIN_FEATURES = [
    'Age', 'Relationship with husband', 'Relationship with the newborn',
    'Feeling about motherhood', 'Recieved Support', 'Need for Support', 'Abuse',
    'Trust and share feelings', 'Worry about newborn',
    'Relax/sleep when newborn is tended ', 'Relax/sleep when the newborn is asleep',
    'Angry after latest child birth', 'Feeling for regular activities',
    'Depression before pregnancy (PHQ2)', 'Depression during pregnancy (PHQ2)',
]
CX_train, CX_val, CX_test = X_train[CHECKIN_FEATURES], X_val[CHECKIN_FEATURES], X_test[CHECKIN_FEATURES]
checkin_models, checkin_results = {}, {}
for name, estimator in model_definitions().items():
    # Match the deployed tree depth while preserving the same seven algorithm families.
    if name == 'Decision Tree':
        estimator = DecisionTreeClassifier(max_depth=7, min_samples_leaf=5, class_weight='balanced', random_state=SEED)
    pipeline = make_pipeline(estimator, CHECKIN_FEATURES)
    start = time.perf_counter(); pipeline.fit(CX_train, y_train); elapsed = time.perf_counter() - start
    result = calculate_metrics(pipeline, CX_val, y_val); result['fit_seconds'] = elapsed
    checkin_models[name], checkin_results[name] = pipeline, result

checkin_comparison = pd.DataFrame({name: {
    'accuracy': value['accuracy'], 'precision': value['precision'],
    'recall': value['recall'], 'f1_score': value['f1_score'], 'fit_seconds': value['fit_seconds']
} for name, value in checkin_results.items()}).T
display(checkin_comparison.sort_values('f1_score', ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
checkin_comparison[['accuracy', 'precision', 'recall', 'f1_score']].plot.bar(
    ax=axes[0], color=['#4f6d7a', '#d8a48f', '#704f6f', '#8fb996'])
axes[0].set(title='Check-in: validation metrics', ylabel='Score', ylim=(0, 1), xlabel='Model')
axes[0].tick_params(axis='x', rotation=35)
checkin_comparison['fit_seconds'].sort_values().plot.barh(ax=axes[1], color='#4f6d7a')
axes[1].set(title='Check-in training time', xlabel='Seconds', ylabel='Model')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(17, 9))
for ax, (name, result) in zip(axes.flat, checkin_results.items()):
    matrix = result['confusion_matrix']; ax.imshow(matrix, cmap='Purples')
    for row in range(2):
        for col in range(2): ax.text(col, row, matrix[row, col], ha='center', va='center', fontsize=12)
    ax.set_xticks([0, 1], ['Elevated', 'Not elevated']); ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
    ax.set(title=name, xlabel='Predicted', ylabel='Actual')
axes.flat[-1].axis('off')
fig.suptitle('Validation confusion matrix for every check-in candidate', fontsize=15)
plt.tight_layout(); plt.show()

**Interpretation:** With only 15 user-friendly inputs, Logistic Regression has the best validation F1. Performance remains stronger than the dummy baseline, while the confusion matrices expose the different trade-offs between missed elevated cases and false alerts.

In [ ]:
checkin_winner_name = checkin_comparison['f1_score'].idxmax()
checkin_winner = make_pipeline(model_definitions()[checkin_winner_name], CHECKIN_FEATURES)
checkin_winner.fit(pd.concat([CX_train, CX_val]), pd.concat([y_train, y_val]))
checkin_test = calculate_metrics(checkin_winner, CX_test, y_test)
display(pd.Series({k: v for k, v in checkin_test.items() if k != 'confusion_matrix'},
                  name='untouched test score').round(4).to_frame())

cm2 = checkin_test['confusion_matrix']
fig, ax = plt.subplots(figsize=(5, 4)); ax.imshow(cm2, cmap='Greens')
for row in range(2):
    for col in range(2): ax.text(col, row, cm2[row, col], ha='center', va='center', fontsize=15)
ax.set_xticks([0, 1], ['Elevated', 'Not elevated']); ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
ax.set(title=f'Check-in untouched test: {checkin_winner_name}', xlabel='Predicted', ylabel='Actual')
plt.show()

# Logistic Regression coefficients: magnitude indicates stronger influence, not causation.
coef_names = checkin_winner.named_steps['preprocess'].get_feature_names_out()
coefs = checkin_winner.named_steps['model'].coef_[0]
coef_table = pd.Series(coefs, index=coef_names).sort_values(key=abs, ascending=False).head(20)
ax = coef_table.sort_values().plot.barh(figsize=(10, 7), color=['#d8a48f' if x < 0 else '#704f6f' for x in coef_table.sort_values()])
ax.set(title='Check-in Logistic Regression: 20 largest coefficients', xlabel='Coefficient', ylabel='Transformed predictor')
plt.tight_layout(); plt.show()

**Interpretation:** The reduced check-in correctly identifies 43 of 53 elevated test cases but creates more false alerts than the full model. Positive and negative coefficient directions describe associations with the model's encoded class; their magnitude reflects influence, not medical causation.

## 9. Model Export

Saving occurs only after the experiment is complete. The application loads these fitted pipelines; it does not retrain models for every user request.

In [ ]:
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump(full_winner, ROOT / 'models' / 'ppd_screening_risk.joblib')
joblib.dump(checkin_winner, ROOT / 'models' / 'ppd_checkin_risk.joblib')

summary = pd.DataFrame({
    'Full screening model': [full_winner_name, *[full_test[k] for k in ['accuracy', 'precision', 'recall', 'f1_score']]],
    'Reduced check-in model': [checkin_winner_name, *[checkin_test[k] for k in ['accuracy', 'precision', 'recall', 'f1_score']]],
}, index=['selected model', 'accuracy', 'precision', 'recall', 'f1_score'])
display(summary)

## 10. Bilingual Retrieval

The 800 tabular rows train screening classifiers; they do not contain conversational answers. The chatbot therefore uses a separate 14-topic, source-attributed English/Kinyarwanda knowledge collection. The Kinyarwanda renderings require native-speaker review.

At runtime: user text → deterministic safety/scope checks → language selection → TF-IDF retrieval → optional local Ollama phrasing → disclaimer and sources. The Ollama model is pretrained and Modelfile customization is not fine-tuning.

In [ ]:
knowledge = json.loads(KNOWLEDGE.read_text(encoding='utf-8'))
print('Knowledge topics:', len(knowledge))
display(pd.DataFrame([{
    'topic': row['topic'], 'source': row['source'], 'has English': bool(row.get('text_en')),
    'has Kinyarwanda': bool(row.get('text_rw')), 'review status': row.get('review_status', '')
} for row in knowledge]))

retrievers = {}
for language in ['en', 'rw']:
    qkey, textkey = f'queries_{language}', f'text_{language}'
    searchable = [f"{row.get(qkey, '')} {row.get(textkey, '')}" for row in knowledge]
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)
    matrix = vectorizer.fit_transform(searchable)
    retrievers[language] = (vectorizer, matrix)

def retrieve(query, language='en', top_k=3):
    vectorizer, matrix = retrievers[language]
    similarities = cosine_similarity(vectorizer.transform([query]), matrix)[0]
    indexes = similarities.argsort()[::-1][:top_k]
    return [(knowledge[i], float(similarities[i])) for i in indexes]

queries = [
    ('I cannot sleep and I feel exhausted', 'en'),
    ('I feel guilty and like a bad mother', 'en'),
    ('Sinshobora gusinzira kandi ndananiwe', 'rw'),
    ('Numva mfite agahinda nyuma yo kubyara', 'rw'),
]
retrieval_results = []
for query, language in queries:
    row, score = retrieve(query, language, 1)[0]
    retrieval_results.append({'query': query, 'language': language, 'retrieved topic': row['topic'], 'similarity': score})
display(pd.DataFrame(retrieval_results).round(3))

## 11. Generator Dataset

The response generator is evaluated separately from classification and retrieval. It uses
`google/mt5-small` with a LoRA adapter and evidence-conditioned supervised examples.

- Six prompt forms per language are applied to each of 14 reviewed topics.
- Complete topics—not paraphrases—are assigned to train, validation, or test.
- The 24 test examples come from two topics unseen during training and validation.
- The 800 participant rows are never converted into chatbot answers.
- Automatic metrics assess overlap and grounding; human review remains necessary for empathy,
  fluency, safety, and native Kinyarwanda correctness.


In [ ]:
import torch

RUN_GENERATOR_TRAINING = True
BASE_MODEL = 'google/mt5-small'
GENERATOR_DIR = MODEL_DIR / 'umubyeyi-mt5-lora'
PROMPTS = {
    'en': [
        'I need emotional support with {terms}.',
        'After giving birth, I have been struggling with {terms}.',
        'Can you help me understand {terms}?',
        'I am a new mother dealing with {terms}.',
        'What can I do when I experience {terms}?',
        'Please support me with {terms}.',
    ],
    'rw': [
        'Nkeneye ubufasha ku bijyanye na {terms}.',
        'Nyuma yo kubyara ndimo guhangana na {terms}.',
        'Wamfasha gusobanukirwa {terms}?',
        'Ndi umubyeyi mushya mpanganye na {terms}.',
        'Nakora iki iyo mfite {terms}?',
        'Mfasha ku bijyanye na {terms}.',
    ],
}
ACK = {
    'en': ['Thank you for sharing this.', 'I hear that this is difficult.', 'You are not alone in facing this.'],
    'rw': ['Urakoze kubivuga.', 'Ndumva ko ibi bikugoye.', 'Nturi wenyine muri ibi.'],
}

def format_generator_input(query, evidence, language):
    language_name = 'Kinyarwanda' if language == 'rw' else 'English'
    return (
        'Generate an empathetic postpartum emotional-support answer. Use only the evidence, '
        'address the mother directly, and do not diagnose.\n'
        f'Language: {language_name}\nEvidence: {evidence.strip()}\n'
        f'Mother: {query.strip()}\nAnswer:'
    )

bank = json.loads(KNOWLEDGE.read_text(encoding='utf-8'))
topic_ids = sorted(row['id'] for row in bank)
random.Random(SEED).shuffle(topic_ids)
topic_split = {
    topic: ('test' if index < 2 else 'validation' if index < 4 else 'train')
    for index, topic in enumerate(topic_ids)
}

generator_examples = []
for row in bank:
    for language in ('en', 'rw'):
        evidence = row[f'text_{language}'].strip()
        terms = row[f'queries_{language}'].strip()
        for variant, template in enumerate(PROMPTS[language]):
            query = template.format(terms=terms)
            generator_examples.append({
                'topic_id': row['id'], 'topic': row['topic'], 'language': language,
                'split': topic_split[row['id']], 'query': query, 'evidence': evidence,
                'input': format_generator_input(query, evidence, language),
                'target': f"{ACK[language][variant % len(ACK[language])]} {evidence}",
            })

generator_frame = pd.DataFrame(generator_examples)
display(generator_frame.groupby(['split', 'language']).size().unstack(fill_value=0))
display(pd.DataFrame({'topic_id': topic_ids, 'split': [topic_split[t] for t in topic_ids]}))
assert len(generator_examples) == 168
assert set(generator_frame.query("split == 'train'").topic_id).isdisjoint(
    set(generator_frame.query("split == 'test'").topic_id)
)

split_counts = generator_frame.groupby(['split', 'language']).size().unstack(fill_value=0)
ax = split_counts.reindex(['train', 'validation', 'test']).plot.bar(
    figsize=(8, 5), color=['#4f6d7a', '#a77887'], title='Generator examples by split and language'
)
ax.set(xlabel='Split', ylabel='Examples'); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
if RUN_GENERATOR_TRAINING:
    assert torch.cuda.is_available(), 'Enable a T4 GPU in Colab before running generator training.'


## 12. LoRA Fine-Tuning

The base model is evaluated on the untouched-topic test set before any adapter updates. Only
the LoRA parameters are trained. Validation loss selects the best checkpoint, with early
stopping available if improvement stalls.


In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
    EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_generator_batch(batch):
    encoded = tokenizer(batch['input'], max_length=512, truncation=True)
    encoded['labels'] = tokenizer(
        text_target=batch['target'], max_length=220, truncation=True
    )['input_ids']
    return encoded

generator_datasets = {}
for split_name in ('train', 'validation'):
    rows = generator_frame.query('split == @split_name')[['input', 'target']].to_dict('records')
    generator_datasets[split_name] = Dataset.from_list(rows).map(
        tokenize_generator_batch, batched=True, remove_columns=['input', 'target']
    )

base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16,
    lora_dropout=0.05, target_modules=['q', 'v'], bias='none'
)
generator_model = get_peft_model(base_model, lora_config)
generator_model.to('cuda')
trainable_parameters, total_parameters = generator_model.get_nb_trainable_parameters()
print(
    f'Trainable parameters: {trainable_parameters:,}/{total_parameters:,} '
    f'({100 * trainable_parameters / total_parameters:.4f}%)'
)

test_rows = generator_frame.query("split == 'test'").to_dict('records')

def generate_test_predictions(current_model):
    predictions = []
    current_model.eval()
    for row in test_rows:
        batch = tokenizer(
            row['input'], return_tensors='pt', truncation=True, max_length=512
        ).to(current_model.device)
        with torch.inference_mode():
            output = current_model.generate(
                **batch, max_new_tokens=180, num_beams=2, no_repeat_ngram_size=3
            )
        predictions.append(tokenizer.decode(output[0], skip_special_tokens=True).strip())
    return predictions

print('Generating untouched-topic predictions from the base model...')
baseline_predictions = generate_test_predictions(generator_model)

training_arguments = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_ROOT / 'generator_checkpoints'),
    num_train_epochs=12, learning_rate=1e-3,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, eval_strategy='epoch', save_strategy='epoch',
    logging_steps=5, save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model='eval_loss', greater_is_better=False,
    report_to='none', fp16=True, seed=SEED,
)
generator_trainer = Seq2SeqTrainer(
    model=generator_model, args=training_arguments,
    train_dataset=generator_datasets['train'],
    eval_dataset=generator_datasets['validation'],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=generator_model),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
training_started = time.time()
training_result = generator_trainer.train()
print('Training minutes:', round((time.time() - training_started) / 60, 2))


## 13. Generator Evaluation

ROUGE-L measures sequence overlap with the reference answer. Grounding overlap measures how
much generated wording is supported by the evidence. The strict gate requires substantial
reproduction of complete evidence sentences and is reported separately for English and
Kinyarwanda. These metrics do not replace human review.


In [ ]:
log_history = generator_trainer.state.log_history
train_log = pd.DataFrame([
    {'step': row['step'], 'epoch': row.get('epoch'), 'loss': row['loss']}
    for row in log_history if 'loss' in row
])
eval_log = pd.DataFrame([
    {'step': row['step'], 'epoch': row.get('epoch'), 'eval_loss': row['eval_loss']}
    for row in log_history if 'eval_loss' in row
])
display(eval_log.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_log['step'], train_log['loss'], marker='o', color='#4f6d7a')
axes[0].set(title='LoRA training loss', xlabel='Training step', ylabel='Loss')
axes[1].plot(eval_log['epoch'], eval_log['eval_loss'], marker='o', color='#704f6f')
axes[1].set(title='Validation loss by epoch', xlabel='Epoch', ylabel='Validation loss')
axes[1].set_xticks(eval_log['epoch'])
plt.tight_layout(); plt.show()


In [ ]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def grounding_overlap(answer, evidence):
    words = re.findall(r'[^\W\d_]{3,}', answer.lower(), flags=re.UNICODE)
    evidence_words = set(re.findall(r'[^\W\d_]{3,}', evidence.lower(), flags=re.UNICODE))
    return sum(word in evidence_words for word in words) / len(words) if words else 0.0

def strict_grounding_accept(answer, evidence):
    normalize = lambda text: ' '.join(re.findall(r'[^\W_]+', text.lower(), flags=re.UNICODE))
    sentences = [
        sentence.strip() for sentence in re.split(r'(?<=[.!?])\s+', evidence)
        if sentence.strip()
    ]
    matched = [sentence for sentence in sentences if normalize(sentence) in normalize(answer)]
    matched_words = sum(len(normalize(sentence).split()) for sentence in matched)
    evidence_words = sum(len(normalize(sentence).split()) for sentence in sentences)
    return matched_words / max(1, evidence_words) >= 0.65

def generator_metrics(predictions):
    rouge_l = np.mean([
        rouge.score(row['target'], prediction)['rougeL'].fmeasure
        for row, prediction in zip(test_rows, predictions)
    ])
    overlap = np.mean([
        grounding_overlap(prediction, row['evidence'])
        for row, prediction in zip(test_rows, predictions)
    ])
    acceptance = {}
    for language in ('en', 'rw'):
        pairs = [
            (row, prediction) for row, prediction in zip(test_rows, predictions)
            if row['language'] == language
        ]
        acceptance[language] = np.mean([
            strict_grounding_accept(prediction, row['evidence']) for row, prediction in pairs
        ])
    return {
        'rouge_l_f1': float(rouge_l),
        'mean_grounding_overlap': float(overlap),
        'english_strict_acceptance': float(acceptance['en']),
        'kinyarwanda_strict_acceptance': float(acceptance['rw']),
        'examples': len(test_rows),
    }

fine_tuned_predictions = generate_test_predictions(generator_model)
baseline_generator_metrics = generator_metrics(baseline_predictions)
fine_tuned_generator_metrics = generator_metrics(fine_tuned_predictions)
generator_comparison = pd.DataFrame(
    [baseline_generator_metrics, fine_tuned_generator_metrics],
    index=['Base mT5', 'Fine-tuned mT5']
)
display(generator_comparison.round(4))

plot_columns = [
    'rouge_l_f1', 'mean_grounding_overlap',
    'english_strict_acceptance', 'kinyarwanda_strict_acceptance'
]
ax = generator_comparison[plot_columns].plot.bar(
    figsize=(12, 6), color=['#4f6d7a', '#d8a48f', '#704f6f', '#8fb996'],
    title='Base versus fine-tuned generator on untouched topics'
)
ax.set(xlabel='Model', ylabel='Score', ylim=(0, 1)); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

review_table = pd.DataFrame([{
    'topic': row['topic'], 'language': row['language'], 'question': row['query'],
    'reference': row['target'], 'base_prediction': base_prediction,
    'fine_tuned_prediction': prediction,
    'strict_accept': strict_grounding_accept(prediction, row['evidence']),
    'reviewer': '', 'empathy_1_to_5': '', 'fluency_1_to_5': '',
    'safety_1_to_5': '', 'language_correctness_1_to_5': '', 'review_notes': '',
} for row, base_prediction, prediction in zip(
    test_rows, baseline_predictions, fine_tuned_predictions
)])
display(review_table)


## 14. Artifact Export

The archive includes the fitted classical models, LoRA adapter, tokenizer, training manifest,
raw held-out generations, metric tables, and loss history. Keep the archive with the exact
executed notebook used for the defense.


In [ ]:
# Save classical models produced above.
joblib.dump(full_winner, MODEL_DIR / 'ppd_screening_risk.joblib')
joblib.dump(checkin_winner, MODEL_DIR / 'ppd_checkin_risk.joblib')

GENERATOR_DIR.mkdir(parents=True, exist_ok=True)
generator_model.save_pretrained(GENERATOR_DIR)
tokenizer.save_pretrained(GENERATOR_DIR)

accepted_languages = [
    language for language, column in (
        ('en', 'english_strict_acceptance'), ('rw', 'kinyarwanda_strict_acceptance')
    ) if fine_tuned_generator_metrics[column] >= 0.5
]
generator_manifest = {
    'fine_tuned': True,
    'method': 'LoRA supervised fine-tuning (PEFT) in unified Colab notebook',
    'base_model': BASE_MODEL,
    'seed': SEED,
    'dataset': {
        'examples': len(generator_examples), 'topics': len(bank),
        'splits': generator_frame['split'].value_counts().to_dict(),
        'languages': generator_frame['language'].value_counts().to_dict(),
        'topics_by_split': {
            split_name: sorted(generator_frame.query('split == @split_name').topic_id.unique().tolist())
            for split_name in ('train', 'validation', 'test')
        },
    },
    'trainable_parameters': trainable_parameters,
    'total_parameters': total_parameters,
    'epochs_requested': 12,
    'training_loss': float(training_result.training_loss),
    'baseline_test': baseline_generator_metrics,
    'fine_tuned_test': fine_tuned_generator_metrics,
    'accepted_generation_languages': accepted_languages,
    'limitation': 'Human review is required for empathy, safety, fluency, and Kinyarwanda correctness.',
}
(GENERATOR_DIR / 'training_manifest.json').write_text(
    json.dumps(generator_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
(GENERATOR_DIR / 'test_generations.json').write_text(
    review_table.to_json(orient='records', force_ascii=False, indent=2), encoding='utf-8'
)
(OUTPUT_ROOT / 'generator_loss_history.json').write_text(
    json.dumps(log_history, indent=2), encoding='utf-8'
)
full_comparison.to_csv(OUTPUT_ROOT / 'full_model_validation_comparison.csv')
checkin_comparison.to_csv(OUTPUT_ROOT / 'checkin_validation_comparison.csv')
generator_comparison.to_csv(OUTPUT_ROOT / 'generator_test_comparison.csv')
review_table.to_csv(OUTPUT_ROOT / 'generator_human_review_cases.csv', index=False)

archive_base = Path('/content/umubyeyi_colab_results') if Path('/content').exists() else ROOT / 'umubyeyi_colab_results'
archive = shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT)
print('Saved archive:', archive)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive)


## 15. System Architecture

| Component | Method | Trained by this project? |
|---|---|---|
| Full screening model | Seven algorithms compared; Random Forest selected | Yes |
| Reduced check-in model | Seven algorithms compared; Logistic Regression selected | Yes |
| Language detector | Existing project classifier | Previously trained |
| Retrieval | Character TF-IDF over 14 evidence topics | Fitted/indexed, not generative training |
| Response phrasing | Local `umubyeyi` Ollama model based on Gemma 3 4B | No; pretrained and instruction-customized |
| Safety routing | Deterministic rules | Authored, not trained |

`/api/chat` is only the local application endpoint connecting the browser to this pipeline. It is not an external model and does not train anything.

## 16. Conclusions and Limitations

- Classical training is fast because there are only 560 training rows and 15–46 predictors; this is fundamentally different from language-model training.
- Random Forest full-model test results: accuracy 0.8083, precision 0.7679, recall 0.8113, F1 0.7890.
- Logistic Regression check-in test results: accuracy 0.7667, precision 0.7049, recall 0.8113, F1 0.7544.
- Results are held-out experimental performance, not diagnosis or clinical validation.
- The data is from Bangladesh, covers up to 24 months postpartum, and is not limited to first-time mothers.
- The chatbot requires a separate reviewed conversational dataset before defensible language-model fine-tuning can be claimed.